# Tickets, TF-IDF, and the Network Tuning Cup

Each row is a synthetic support ticket. The label is `category`: delivery, payment, product, or account. TF-IDF turns the text into numbers. An MLP then classifies those numbers.

Every fit is logged to a local MLflow file store in `tracking/`. There is no server. The best validation accuracy wins. If two runs tie, the faster one ranks first.

The cup budget is 40 iterations. A network can still be climbing when the budget ends. That is why a higher learning rate can beat a deeper net on this table.

The stretch cell logs a confusion matrix on the best run.


In [ ]:
import os
import sys
from pathlib import Path

os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"

here = Path.cwd().resolve()
root = None
for candidate in [here, *here.parents]:
    for folder in (candidate, candidate / "04-deep-learning"):
        marker = folder / "lib" / "customers.py"
        if marker.is_file() and (folder / "app.py").is_file():
            root = folder
            break
    if root is not None:
        break
if root is None:
    raise FileNotFoundError("Start Jupyter from the repo folder or from 04-deep-learning.")
lib = root / "lib"
if str(lib) not in sys.path:
    sys.path.insert(0, str(lib))
ROOT = root
print("module", ROOT)


In [ ]:
from tickets import confusion_frame, fit_text, load_tickets, split_tickets
from tracking import leaderboard, log_confusion_csv, log_cup_run

tickets = load_tickets()
train, validation = split_tickets(tickets)
print(tickets["category"].value_counts().to_string())
print("train", len(train), "validation", len(validation))


## Cup

Exercise. Add one more dict to `configs`, then run the cell again. Each run is kept. Re-running the cell adds another copy of the same settings.

Engineer stretch: add a wider network, such as `(128, 64)`, and compare `training_seconds` on the leaderboard.


In [ ]:
configs = [
    {"hidden_layer_sizes": (16,), "learning_rate_init": 0.001},
    {"hidden_layer_sizes": (32,), "learning_rate_init": 0.001},
    {"hidden_layer_sizes": (32, 16), "learning_rate_init": 0.001},
    {"hidden_layer_sizes": (32,), "learning_rate_init": 0.01},
]
# Exercise. Add one more dict to configs, then run this cell again.

results = []
for config in configs:
    result = fit_text(train, validation, **config)
    result["run_id"] = log_cup_run(ROOT, result)
    results.append(result)
    print(
        result["hidden_layer_sizes"],
        f"{result['learning_rate_init']:.3f}",
        f"{result['validation_accuracy']:.3f}",
        f"{result['training_seconds']:.3f}",
    )

board = leaderboard(ROOT)
print(
    board.drop(columns=["run_id"]).to_string(
        index=False,
        formatters={
            "learning_rate_init": "{:.3f}".format,
            "validation_accuracy": "{:.3f}".format,
            "training_seconds": "{:.3f}".format,
        },
    )
)


## Stretch

Log the confusion matrix of the best run as an MLflow artifact. Rows are the true category. Columns are the predicted category.


In [ ]:
best = max(results, key=lambda item: (item["validation_accuracy"], -item["training_seconds"]))
matrix = confusion_frame(validation, best["predicted"])
print(matrix.to_string())
log_confusion_csv(ROOT, best["run_id"], matrix, ROOT / "tracking" / "confusion.csv")
print("logged", best["run_id"])


## Debrief

Exercise. Which settings won, and what was the validation accuracy?


In [ ]:
debrief = ""  # Exercise. One sentence.
print(debrief or "Write one sentence, then run this cell again.")
